# TripoSG golden set (5 images, GPU)

Run on Colab with a GPU runtime (**Runtime → Change runtime type → T4 GPU** or
better; TripoSG needs >= 8 GB VRAM). You run this; Muse does not.

What it does:

1. Clones TripoSG at a pinned commit and installs only the missing packages.
2. Downloads the `VAST-AI/TripoSG` weights at a recorded revision.
3. Cuts 5 TripoSG example images out to RGBA with `rembg` (u2net). RMBG-1.4 is
   skipped for the MVP, and TripoSG only bypasses RMBG when the input has a
   real alpha channel, which none of the repo examples have.
4. Runs the official path (`prepare_image(..., rmbg_net=None)` then the
   pipeline) at FP32 with fixed seeds.
5. Saves, per image: the RGBA cut-out, the prepared input, the DINOv2 input and
   output, the inputs and output of the first transformer step, the final
   latents, and the mesh (`.glb`).
6. Zips everything with a `report.json` and downloads it.

Weights stay in Colab. Bring back the zip; only `report.json` and hashes go
into git (10 MB cap).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/VAST-AI-Research/TripoSG.git /content/TripoSG
%cd /content/TripoSG
!git checkout --quiet fc5c40990181e2a756c4e0b1c2f4d6b5202faf8c
!git rev-parse HEAD

name, memory.total [MiB]
Tesla T4, 15360 MiB
Cloning into '/content/TripoSG'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 167 (delta 65), reused 52 (delta 52), pack-reused 72 (from 1)
Receiving objects: 100% (167/167), 15.66 MiB | 19.82 MiB/s, done.
Resolving deltas: 100% (79/79), done.
/content/TripoSG
fc5c40990181e2a756c4e0b1c2f4d6b5202faf8c


In [2]:
# Colab already ships torch, diffusers, transformers, numpy, opencv,
# scikit-image and huggingface_hub. TripoSG's requirements.txt pins
# numpy==1.22.3, which has no wheel for Colab's Python, so install only
# what is missing. pymeshlab is skipped: no face reduction here.
# diso compiles a CUDA extension against Colab's torch (takes minutes).
# rembg no longer bundles onnxruntime; [cpu] adds it (5 small images).
!pip install --quiet einops omegaconf jaxtyping typeguard peft trimesh "rembg[cpu]"
!pip install --quiet diso --no-build-isolation

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
  Preparing metadata (pyproject.toml) ... done


In [3]:
import hashlib
import json
import os
import shutil
import sys
import time
from importlib.metadata import version

import numpy as np
import torch
from PIL import Image

assert torch.cuda.is_available(), "Switch the runtime to a GPU first."

TRIPOSG_DIR = "/content/TripoSG"
TRIPOSG_COMMIT = "fc5c40990181e2a756c4e0b1c2f4d6b5202faf8c"
WEIGHTS_REPO = "VAST-AI/TripoSG"
WEIGHTS_DIR = os.path.join(TRIPOSG_DIR, "pretrained_weights/TripoSG")
OUT_DIR = "/content/goldens_5"
DEVICE = "cuda"
# FP32 is the parity reference; the official script runs float16.
DTYPE = torch.float32
STEPS = 50
GUIDANCE = 7.0
WHITE = np.array([1.0, 1.0, 1.0])

MANIFEST = {
    "size": 5,
    "triposg_commit": "fc5c40990181e2a756c4e0b1c2f4d6b5202faf8c",
    "pipeline": "official prepare_image(rmbg_net=None) + TripoSGPipeline",
    "inputs": "rembg/u2net RGBA cut-outs, fed to the phone as-is",
    "images": [
        {
            "id": "hjswed",
            "source": "assets/example_data/hjswed.png",
            "license": "TripoSG repo asset; check before shipping",
            "seed": 42
        },
        {
            "id": "iuvbww",
            "source": "assets/example_data/iuvbww.png",
            "license": "TripoSG repo asset; check before shipping",
            "seed": 43
        },
        {
            "id": "qwvfsd",
            "source": "assets/example_data/qwvfsd.png",
            "license": "TripoSG repo asset; check before shipping",
            "seed": 44
        },
        {
            "id": "uegvsd",
            "source": "assets/example_data/uegvsd.png",
            "license": "TripoSG repo asset; check before shipping",
            "seed": 45
        },
        {
            "id": "yuyyuw",
            "source": "assets/example_data/yuyyuw.png",
            "license": "TripoSG repo asset; check before shipping",
            "seed": 46
        }
    ]
}

os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(OUT_DIR, "manifest_5.json"), "w") as f:
    json.dump(MANIFEST, f, indent=2)

In [4]:
from huggingface_hub import HfApi, snapshot_download

weights_revision = HfApi().model_info(WEIGHTS_REPO).sha
snapshot_download(
    repo_id=WEIGHTS_REPO, revision=weights_revision, local_dir=WEIGHTS_DIR
)

weight_files = []
for root, _, names in os.walk(WEIGHTS_DIR):
    if "/.cache" in root:
        continue
    for name in sorted(names):
        path = os.path.join(root, name)
        rel = os.path.relpath(path, WEIGHTS_DIR)
        weight_files.append({"path": rel, "bytes": os.path.getsize(path)})
        print(f"{os.path.getsize(path):>14,}  {rel}")
print("weights revision", weights_revision)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

         1,564  .gitattributes
         1,465  README.md
           525  model_index.json
           191  transformer/config.json
 5,758,280,104  transformer/diffusion_pytorch_model.safetensors
           436  feature_extractor_dinov2/preprocessor_config.json
           335  vae/config.json
   970,685,468  vae/diffusion_pytorch_model.safetensors
         1,104  image_encoder_dinov2/config.json
 1,217,522,888  image_encoder_dinov2/model.safetensors
           158  scheduler/scheduler_config.json
weights revision 2c1c516d22d58db486a058d98d31bb6177344e06


In [5]:
from rembg import new_session, remove


def has_valid_alpha(rgba, min_ratio=0.01):
    # Same rule as TripoSG's is_valid_alpha: >= 1% of pixels in the lowest and
    # highest of 20 histogram bins. Otherwise TripoSG falls back to RMBG.
    alpha = np.asarray(rgba.getchannel("A"))
    hist, _ = np.histogram(alpha, bins=20, range=(0, 256))
    return min(hist[0], hist[-1]) >= alpha.size * min_ratio


session = new_session("u2net")
for item in MANIFEST["images"]:
    item_dir = os.path.join(OUT_DIR, item["id"])
    os.makedirs(item_dir, exist_ok=True)
    source = Image.open(os.path.join(TRIPOSG_DIR, item["source"])).convert("RGB")
    cutout = remove(source, session=session)
    assert has_valid_alpha(cutout), f"{item['id']}: cut-out alpha is unusable"
    cutout.save(os.path.join(item_dir, "input_rgba.png"))
    print(item["id"], "cut out", cutout.size)

  0%|                                               | 0.00/176M [00:00<?, ?B/s]

hjswed cut out (2048, 2048)
iuvbww cut out (800, 800)
qwvfsd cut out (1024, 1024)
uegvsd cut out (1024, 1024)
yuyyuw cut out (2048, 2048)


In [6]:
sys.path.insert(0, TRIPOSG_DIR)
sys.path.insert(0, os.path.join(TRIPOSG_DIR, "scripts"))

# Newer diffusers only loads classes from modules already imported, so
# import every module model_index.json names before from_pretrained.
import triposg.models.autoencoders.autoencoder_kl_triposg  # noqa: E402,F401
import triposg.models.transformers.triposg_transformer  # noqa: E402,F401
import triposg.schedulers.scheduling_rectified_flow  # noqa: E402,F401
from image_process import prepare_image  # noqa: E402
from triposg.pipelines.pipeline_triposg import TripoSGPipeline  # noqa: E402

pipe = TripoSGPipeline.from_pretrained(WEIGHTS_DIR).to(DEVICE, DTYPE)

state = {"dir": None, "step": 0}


def save(name, tensor):
    array = tensor.detach().float().cpu().numpy()
    np.save(os.path.join(state["dir"], f"{name}.npy"), array)


def on_encoder(module, args, output):
    save("dinov2_pixel_values", args[0])
    save("dinov2_last_hidden_state", output.last_hidden_state)


def on_transformer(module, args, kwargs, output):
    if state["step"] == 0:
        save("step0_latent_model_input", args[0])
        save("step0_timestep", args[1])
        save("step0_encoder_hidden_states", kwargs["encoder_hidden_states"])
        save("step0_noise_pred", output[0])
    state["step"] += 1


def on_step_end(pipeline, i, t, tensors):
    if i == pipeline.num_timesteps - 1:
        save("final_latents", tensors["latents"])
    return {}


pipe.image_encoder_dinov2.register_forward_hook(on_encoder)
pipe.transformer.register_forward_hook(on_transformer, with_kwargs=True)

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

There are modules in TripoSGDiTModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in TripoSGVAEModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


In [7]:
import trimesh  # noqa: E402

results = []
for item in MANIFEST["images"]:
    state["dir"] = os.path.join(OUT_DIR, item["id"])
    state["step"] = 0
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()

    image = prepare_image(
        os.path.join(state["dir"], "input_rgba.png"), bg_color=WHITE, rmbg_net=None
    )
    image.save(os.path.join(state["dir"], "prepared_input.png"))
    with torch.no_grad():
        output = pipe(
            image=image,
            generator=torch.Generator(device=pipe.device).manual_seed(item["seed"]),
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE,
            callback_on_step_end=on_step_end,
        ).samples[0]
    # Same mesh construction as scripts/inference_triposg.py.
    mesh = trimesh.Trimesh(
        output[0].astype(np.float32), np.ascontiguousarray(output[1])
    )
    mesh.export(os.path.join(state["dir"], "mesh.glb"))

    files = {}
    for name in sorted(os.listdir(state["dir"])):
        with open(os.path.join(state["dir"], name), "rb") as f:
            files[name] = hashlib.sha256(f.read()).hexdigest()
    results.append(
        {
            "id": item["id"],
            "seed": item["seed"],
            "seconds": round(time.perf_counter() - started, 1),
            "peak_vram_bytes": torch.cuda.max_memory_allocated(),
            "transformer_calls": state["step"],
            "vertices": len(mesh.vertices),
            "faces": len(mesh.faces),
            "sha256": files,
        }
    )
    print(item["id"], results[-1]["seconds"], "s", len(mesh.faces), "faces")

  0%|          | 0/50 [00:00<?, ?it/s]

/content/TripoSG/triposg/inference_utils.py:259: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  sliced = padded[slice_dims]


final grids shape =  torch.Size([505, 505, 505])
hjswed 413.0 s 1322148 faces


  0%|          | 0/50 [00:00<?, ?it/s]

final grids shape =  torch.Size([505, 505, 505])
iuvbww 460.3 s 1610556 faces


  0%|          | 0/50 [00:00<?, ?it/s]

final grids shape =  torch.Size([505, 505, 505])
qwvfsd 530.9 s 2948944 faces


  0%|          | 0/50 [00:00<?, ?it/s]

final grids shape =  torch.Size([505, 505, 505])
uegvsd 442.3 s 1643304 faces


  0%|          | 0/50 [00:00<?, ?it/s]

final grids shape =  torch.Size([505, 505, 505])
yuyyuw 481.3 s 2018952 faces


In [8]:
report = {
    "triposg_commit": TRIPOSG_COMMIT,
    "weights_repo": WEIGHTS_REPO,
    "weights_revision": weights_revision,
    "weight_files": weight_files,
    "gpu": torch.cuda.get_device_name(0),
    "python": sys.version.split()[0],
    "packages": {
        name: version(name)
        for name in ["torch", "diffusers", "transformers", "diso", "rembg", "numpy"]
    },
    "dtype": str(DTYPE),
    "steps": STEPS,
    "guidance_scale": GUIDANCE,
    "background_removal": "rembg u2net (RMBG-1.4 skipped)",
    "items": results,
}
with open(os.path.join(OUT_DIR, "report.json"), "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps({k: v for k, v in report.items() if k != "weight_files"}, indent=2))

archive = shutil.make_archive("/content/goldens_5", "zip", OUT_DIR)
print("archive ready:", archive)
try:
    from google.colab import files as colab_files

    colab_files.download(archive)
except Exception as error:  # VS Code Colab extension: use its Download...
    print("browser download unavailable here:", type(error).__name__)

{
  "triposg_commit": "fc5c40990181e2a756c4e0b1c2f4d6b5202faf8c",
  "weights_repo": "VAST-AI/TripoSG",
  "weights_revision": "2c1c516d22d58db486a058d98d31bb6177344e06",
  "gpu": "Tesla T4",
  "python": "3.13.16",
  "packages": {
    "torch": "2.11.0+cu130",
    "diffusers": "0.42.1",
    "transformers": "5.22.0",
    "diso": "0.1.4",
    "rembg": "2.0.86",
    "numpy": "2.4.6"
  },
  "dtype": "torch.float32",
  "steps": 50,
  "guidance_scale": 7.0,
  "background_removal": "rembg u2net (RMBG-1.4 skipped)",
  "items": [
    {
      "id": "hjswed",
      "seed": 42,
      "seconds": 413.0,
      "peak_vram_bytes": 11659171840,
      "transformer_calls": 50,
      "vertices": 661075,
      "faces": 1322148,
      "sha256": {
        "dinov2_last_hidden_state.npy": "40ac6fd25b56b8fb6cec1ea7718464ae52e37a252acb007519b9577aba5e1e58",
        "dinov2_pixel_values.npy": "bbf1338ea2533dca7ecdd74f39a0884d6ab8b4ff4a545311925d7e954da9360a",
        "final_latents.npy": "0b9114c71bac4dd260a27f50b0a0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Bring back `/content/goldens_5.zip` (VS Code: Colab server explorer → Download…) and unzip it into `ml/triposg-mobile/reference/goldens_5/`
(git-ignored except `report.json`). Give Muse `report.json` plus any errors.